## Holdout Design and Trigger Bias
Julian Hsu
Date Created: 2026-06-18

The purpose of this script is to show how holdouts can lead to correct launch decisions when there is significant variation in the treatment effect over users trigger into an experiment.

In [10]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt


## Load DGP and primitives
We want there to be variation in the impact over triggering time. Let's code this up by introducing a correlation over when users enter an experiment and their treatment effect.

In [30]:
def create_latent_population(T_max=10, N_per_epoch=100):
    ## For each time period, create a experiment population.
    ## Each population will have a treatment effect that scales with the trigger time.
    output_df = pd.DataFrame()
    for t in range(T_max):
        unit_ids = np.arange(N_per_epoch) + t*N_per_epoch
        treatment_assignment = np.random.choice([0,1], size=N_per_epoch, p = [0.5, 0.5])
        ## Hardcode the treatment effect at some given value to make sure things do not get too crazy
        latent_treatment_effect =  np.log(0.1+t) 
        if latent_treatment_effect > 10:
            latent_treatment_effect =10
        else:
            pass

        outcome = np.random.normal(0,1,size=N_per_epoch) + treatment_assignment*latent_treatment_effect
        entry_df = pd.DataFrame(data={'y':outcome, 'w':treatment_assignment, 'id':unit_ids, 't': [t]*N_per_epoch})

        output_df = pd.concat([output_df, entry_df])
    return output_df 

def t_test(data):
    treatment_group = data[data['w'] == 1]['y'].values
    control_group = data[data['w'] == 0]['y'].values
    ttest, pval, dof = sm.stats.ttest_ind(treatment_group, control_group, usevar='unequal')
    diff = treatment_group.mean() - control_group.mean()
    return diff, pval


In [31]:
T_max_value = 20
pe = create_latent_population(T_max = T_max_value)
for t in range(T_max_value):
    results = t_test(pe.loc[pe.t <= t])
    print('{0}: diff={1:5.3f} pval={2:5.3f}'.format(t, results[0], results[1]))

0: diff=-2.272 pval=0.000
1: diff=-1.025 pval=0.000
2: diff=-0.343 pval=0.029
3: diff=-0.005 pval=0.971
4: diff=0.302 pval=0.013
5: diff=0.499 pval=0.000
6: diff=0.718 pval=0.000
7: diff=0.822 pval=0.000
8: diff=0.985 pval=0.000
9: diff=1.047 pval=0.000
10: diff=1.153 pval=0.000
11: diff=1.255 pval=0.000
12: diff=1.319 pval=0.000
13: diff=1.417 pval=0.000
14: diff=1.505 pval=0.000
15: diff=1.621 pval=0.000
16: diff=1.687 pval=0.000
17: diff=1.738 pval=0.000
18: diff=1.773 pval=0.000
19: diff=1.801 pval=0.000
